In [8]:
##from google.colab import drive
#drive.mount('/content/drive')



In [9]:
!pip install tensorflow pillow numpy tqdm scikit-learn


In [10]:
import os, math, numpy as np
from PIL import Image
from tqdm import tqdm

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model

# 1. Load model pretrained
base = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(224,224,3))
print("✅ Model loaded. Feature dim =", base.output_shape[1])

# 2. Chuẩn bị dataset
# Đường dẫn dataset cho môi trường local
# dataset_root = r"C:\Users\admin\Downloads\Fruit\dataset"
dataset_root = "c:/Users/admin/Downloads/Fruit/dataset"
exts = (".jpg")

paths, labels = [], []
for cls in sorted(os.listdir(dataset_root)):
    cls_dir = os.path.join(dataset_root, cls)
    if not os.path.isdir(cls_dir): continue
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(exts):
            paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

print("Tổng ảnh =", len(paths), " | Số lớp =", len(set(labels)))

# 3. Hàm load + preprocess ảnh
def load_and_preprocess(path, size=(224,224)):
    img = Image.open(path).convert("RGB")
    img = img.resize(size)
    arr = np.array(img, dtype=np.float32)
    return arr

# 4. Trích đặc trưng theo batch
batch_size = 32
features, kept_paths, kept_labels = [], [], []

for i in tqdm(range(math.ceil(len(paths)/batch_size)), desc="Extracting"):
    batch_paths = paths[i*batch_size:(i+1)*batch_size]
    imgs = [load_and_preprocess(p) for p in batch_paths]
    X = np.stack(imgs, axis=0)
    X = preprocess_input(X)
    feats = base.predict(X, verbose=0)
    features.append(feats)
    kept_paths.extend(batch_paths)
    kept_labels.extend(labels[i*batch_size:(i+1)*batch_size])

features = np.vstack(features)
print("✅ Features shape:", features.shape)

# 5. Lưu kết quả
np.save("features.npy", features)
np.save("labels.npy", np.array(kept_labels))
np.save("paths.npy", np.array(kept_paths))


✅ Model loaded. Feature dim = 1280
Tổng ảnh = 1000  | Số lớp = 10


Extracting: 100%|██████████| 32/32 [00:45<00:00,  1.43s/it]

✅ Features shape: (1000, 1280)


In [11]:
import numpy as np
import shutil
import os

# Load dữ liệu
features = np.load("features.npy")
labels = np.load("labels.npy")
paths = np.load("paths.npy")

# Kiểm tra kích thước
print("✅ Features shape:", features.shape)
print("✅ Labels shape:", labels.shape)
print("✅ Paths shape:", paths.shape)

# Xem vài giá trị đầu
print("\nVí dụ 5 vector đầu tiên:")
print(features[:5])

print("\n5 nhãn đầu:")
print(labels[:5])

print("\n5 đường dẫn ảnh đầu:")
print(paths[:5])

# Di chuyển file sang thư mục results/ để backend sử dụng
results_dir = "c:/Users/admin/Downloads/Fruit/results"
shutil.move("features.npy", os.path.join(results_dir, "features.npy"))
shutil.move("labels.npy", os.path.join(results_dir, "labels.npy"))
shutil.move("paths.npy", os.path.join(results_dir, "paths.npy"))
print("\n✅ Đã di chuyển các file features.npy, labels.npy, paths.npy sang thư mục results/")


✅ Features shape: (1000, 1280)
✅ Labels shape: (1000,)
✅ Paths shape: (1000,)

Ví dụ 5 vector đầu tiên:
[[0.49783045 0.04007382 0.         ... 0.01347135 0.7152931  0.        ]
 [0.         0.061095   0.         ... 0.         0.7103991  0.        ]
 [0.61873853 0.5839885  0.         ... 0.         1.5019755  0.01184622]
 [0.79602873 0.63544136 0.         ... 0.         0.49312583 0.        ]
 [0.23319583 0.33710837 0.         ... 0.         0.40508488 0.01389178]]

5 nhãn đầu:
['apple' 'apple' 'apple' 'apple' 'apple']

5 đường dẫn ảnh đầu:
['c:/Users/admin/Downloads/Fruit/dataset\\apple\\apple_001.jpg'
 'c:/Users/admin/Downloads/Fruit/dataset\\apple\\apple_002.jpg'
 'c:/Users/admin/Downloads/Fruit/dataset\\apple\\apple_003.jpg'
 'c:/Users/admin/Downloads/Fruit/dataset\\apple\\apple_004.jpg'
 'c:/Users/admin/Downloads/Fruit/dataset\\apple\\apple_005.jpg']

✅ Đã di chuyển các file features.npy, labels.npy, paths.npy sang thư mục results/
